# Task 1: Verified recursive self-training (Amharic news topic classification)
Loop: generate pseudo-labels -> verify (confidence + independent char-n-gram model) -> retrain a copy -> accept/rollback on **dev** -> repeat. Gold labels of the pool are used only to *measure* pseudo-label precision. The test split is used only in the final cell.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils import load_config
from src.data import load_base, make_split
from src.features import attach_views

DATA_SOURCE = "local"   # "local" (data/raw TSVs), "hf", or "synthetic" (smoke test only)
cfg = load_config(overrides={"data.source": DATA_SOURCE})
base = attach_views(load_base(cfg), cfg)
cfg["_n_classes"] = base.n_classes
print(base.label_names, {k: len(v) for k, v in base.texts.items()})

In [ ]:
from task1_recursive.selftrain import run_selftrain
SEED = 0
split = make_split(base, SEED, cfg['data']['seed_per_class'])
print('seed set:', len(split.seed_idx), 'pool:', len(split.pool_idx))
res = run_selftrain(base, split, cfg, SEED)
hist = pd.DataFrame(res.history)
hist

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist['round'], hist['dev_macro_f1'], marker='o'); ax[0].set_title('Dev macro-F1 per round'); ax[0].set_xlabel('round')
ax[1].plot(hist['round'], hist['pl_precision'], marker='o', color='tab:red'); ax[1].set_title('Pseudo-label precision (vs hidden gold)'); ax[1].set_xlabel('round')
plt.tight_layout(); plt.show()

In [ ]:
from src.eval import scores
Xt = base.X['teacher']; y = base.y['test']
for name, m in [('round-0 teacher', res.teacher_round0), ('final teacher', res.teacher_final)]:
    print(name, {k: round(v, 4) for k, v in scores(m.proba(Xt['test']), y).items()})

Full multi-seed comparison (verified vs confidence-only vs naive): `python -m task1_recursive.run`.